In [15]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import avg, count 

In [2]:
spark = SparkSession.builder.appName("BigDataLab1").getOrCreate() 

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 18:40:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/10 18:40:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
movies_df = spark.read.csv("./data/filme.csv", header=True, inferSchema=True) 
ratings_df = spark.read.csv("./data/ratinguri.csv", header=True, inferSchema=True) 

In [21]:
movies_df.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)



In [22]:
ratings_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)



In [6]:
# Selectarea coloanelor 
movies_df.select("title", "genre").show(5) 

+--------------------+-------+
|               title|  genre|
+--------------------+-------+
|      Umbra Orașului|Actiune|
|Misiune la Miezul...|Actiune|
|       Ultimul Agent|Actiune|
|          Codul Roșu|Actiune|
|     Fuga prin Munți|Actiune|
+--------------------+-------+
only showing top 5 rows


In [ ]:
# Filtrarea rândurilor 
animation_movies = movies_df.filter(movies_df.genre == "Animatie") 

In [8]:
animation_movies.show(5) 

+-------+--------------------+--------+
|movieId|               title|   genre|
+-------+--------------------+--------+
|     16|     Regatul Norilor|Animatie|
|     17|    Micul Inventator|Animatie|
|     18|Prietenii din Pădure|Animatie|
|     19|   Călătorie pe Lună|Animatie|
|     20|   Dragonul Albastru|Animatie|
+-------+--------------------+--------+



In [ ]:
# Filtrarea rating-urilor >= 4.5 
high_ratings = ratings_df.filter(ratings_df.rating >= 4.5) 

In [9]:
high_ratings.show(5) 

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|     17|   4.7|
|     2|      1|   4.8|
|     2|      7|   4.5|
|     3|      7|   4.7|
|     5|      2|   4.5|
+------+-------+------+
only showing top 5 rows


In [ ]:
# Rating mediu per film 
avg_ratings = ratings_df.groupBy("movieId").agg(avg("rating").alias("avg_rating")) 

In [11]:
avg_ratings.show(5) 

+-------+------------------+
|movieId|        avg_rating|
+-------+------------------+
|     12|2.9666666666666663|
|      1|               3.2|
|     13|               2.9|
|      6|               2.4|
|     16|               1.8|
+-------+------------------+
only showing top 5 rows


In [ ]:
# Număr rating-uri per film 
count_ratings = ratings_df.groupBy("movieId").count() 

In [12]:
count_ratings.show(5) 

+-------+-----+
|movieId|count|
+-------+-----+
|     12|    3|
|      1|    2|
|     13|    2|
|      6|    2|
|     16|    1|
+-------+-----+
only showing top 5 rows


In [ ]:
# Join filme și rating-uri 
movie_ratings = movies_df.join(ratings_df, on="movieId", how="inner") 

In [13]:
movie_ratings.show(5) 

+-------+--------------------+-------+------+------+
|movieId|               title|  genre|userId|rating|
+-------+--------------------+-------+------+------+
|      1|      Umbra Orașului|Actiune|     2|   4.8|
|      1|      Umbra Orașului|Actiune|     1|   1.6|
|      2|Misiune la Miezul...|Actiune|     9|   2.7|
|      2|Misiune la Miezul...|Actiune|     6|   1.6|
|      2|Misiune la Miezul...|Actiune|     5|   4.5|
+-------+--------------------+-------+------+------+
only showing top 5 rows


In [ ]:
# Rating mediu per gen 
genre_avg = movie_ratings.groupBy("genre").agg(avg("rating").alias("avg_genre_rating")) 

In [14]:
genre_avg.show() 

+--------+------------------+
|   genre|  avg_genre_rating|
+--------+------------------+
| Comedie| 2.786666666666667|
| Actiune|3.0090909090909093|
|  Darama|2.6076923076923078|
|Animatie|3.0454545454545454|
+--------+------------------+



In [17]:
movies_df.createOrReplaceTempView("movies") 
ratings_df.createOrReplaceTempView("ratings") 

In [ ]:
# Toate filmele de gen "Acțiune" 
action_movies = spark.sql("SELECT * FROM movies WHERE genre = 'Actiune'") 

In [19]:
action_movies.show() 

+-------+--------------------+-------+
|movieId|               title|  genre|
+-------+--------------------+-------+
|      1|      Umbra Orașului|Actiune|
|      2|Misiune la Miezul...|Actiune|
|      3|       Ultimul Agent|Actiune|
|      4|          Codul Roșu|Actiune|
|      5|     Fuga prin Munți|Actiune|
+-------+--------------------+-------+



In [ ]:
# Rating mediu per film 
avg_ratings = spark.sql(""" 
    SELECT r.movieId, m.title, AVG(r.rating) AS avg_rating 
    FROM ratings r 
    JOIN movies m ON r.movieId = m.movieId 
    GROUP BY r.movieId, m.title 
    ORDER BY avg_rating DESC 
""") 

In [20]:
avg_ratings.show() 

+-------+--------------------+------------------+
|movieId|               title|        avg_rating|
+-------+--------------------+------------------+
|     17|    Micul Inventator|               4.7|
|     19|   Călătorie pe Lună|               4.2|
|      5|     Fuga prin Munți|               3.4|
|      4|          Codul Roșu|               3.4|
|      2|Misiune la Miezul...|              3.25|
|      1|      Umbra Orașului|               3.2|
|      7|  Vecinii de la Trei|2.9799999999999995|
|     12|   Dincolo de Tăcere|2.9666666666666663|
|     15|  Oameni și Amintiri|2.9333333333333336|
|     13|       Ultima Toamnă|               2.9|
|      8|  Nuntă cu Peripeții|             2.875|
|     20|   Dragonul Albastru|             2.725|
|      9|  Șeful din Greșeală|2.6500000000000004|
|     18|Prietenii din Pădure| 2.566666666666667|
|      6| Vacanță cu Surprize|               2.4|
|     11|  Scrisori Netrimise|2.3333333333333335|
|     16|     Regatul Norilor|               1.8|


In [28]:
top_3 = spark.sql("""
    SELECT
        userId,
        genre,
        ROUND(avg_rating, 2) AS avg_rating
    FROM (
        SELECT
            r.userId,
            m.genre,
            AVG(r.rating) AS avg_rating,
            ROW_NUMBER() OVER (
                PARTITION BY r.userId
                ORDER BY AVG(r.rating) DESC
            ) AS position
        FROM ratings r
        JOIN movies m ON r.movieId = m.movieId
        GROUP BY r.userId, m.genre
    )
    WHERE position <= 3
    ORDER BY userId, position
""")

In [29]:
top_3.show(100, truncate=False)

+------+--------+----------+
|userId|genre   |avg_rating|
+------+--------+----------+
|1     |Animatie|4.7       |
|1     |Comedie |2.95      |
|1     |Actiune |1.7       |
|2     |Comedie |4.5       |
|2     |Actiune |3.37      |
|2     |Darama  |2.4       |
|3     |Comedie |3.35      |
|3     |Animatie|2.7       |
|3     |Darama  |2.35      |
|4     |Actiune |3.4       |
|4     |Animatie|3.2       |
|4     |Darama  |1.6       |
|5     |Actiune |4.75      |
|5     |Darama  |3.1       |
|5     |Comedie |1.5       |
|6     |Darama  |2.8       |
|6     |Animatie|2.4       |
|6     |Actiune |2.0       |
|7     |Darama  |2.77      |
|7     |Comedie |2.3       |
|8     |Comedie |4.2       |
|8     |Animatie|3.7       |
|9     |Darama  |3.5       |
|9     |Comedie |3.0       |
|9     |Actiune |2.7       |
|10    |Animatie|2.85      |
|10    |Comedie |2.6       |
|10    |Darama  |2.2       |
+------+--------+----------+



In [30]:
spark.stop()